# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library. All dataset entities, such as record sets and fields, are referenced by their `@id` for consistency and traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

**Domain:** Human protein quantification by mass spectrometry from mast cell extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` is installed. Restart the kernel after installation if running interactively.
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
print(f"Dataset loaded: {ds.metadata.name}\n")
print(f"Description: {ds.metadata.description}\n")
print(f"Published: {ds.metadata.datePublished if hasattr(ds.metadata, 'datePublished') else 'n/a'}")
print(f"License: {ds.metadata.license if hasattr(ds.metadata, 'license') else 'n/a'}")
print(f"Keywords: {ds.metadata.keywords if hasattr(ds.metadata, 'keywords') else 'n/a'}")

## 2. Data Overview
The dataset may provide multiple _record sets_, each described by a unique `@id`. Below we enumerate all available record sets and their fields (by `@id`).

In [ ]:
# Get the list of all record sets in the dataset by @id
record_sets = [r['@id'] for r in ds.metadata.record_sets]
print('Record Sets and their fields:')
for recset in ds.metadata.record_sets:
    recset_id = recset['@id']
    print(f"\nRecord Set @id: {recset_id}")
    name = recset['name'] if 'name' in recset else ''
    desc = recset.get('description', '')
    print(f"  Name: {name}")
    print(f"  Description: {desc}")
    if 'fields' in recset:
        print("  Fields:")
        for field in recset['fields']:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - {field_id}")
    else:
        print("  No fields listed.")

## 3. Data Extraction
Here we load the data from a specific record set into a DataFrame for inspection and analysis. **All interactions with record sets and fields are by their `@id`.**

In [ ]:
# We'll extract data from each record set found above by @id
dataframes = dict()
for recset in ds.metadata.record_sets:
    recset_id = recset['@id']
    print(f'Loading record set: {recset_id}')
    records = list(ds.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

# Choose the first record set for exploration
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing record set '{record_set_id}' for further exploration.")
    display(dataframes[record_set_id].head())
else:
    print('No tabular data found!')

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic filtering, normalization, and grouping. 
All field and column references use their `@id` as per the Croissant schema.

In [ ]:
# Select a numeric field (by @id) for analysis. Replace with the actual field @id found in section 2/3.
# For demonstration, we try several possible column names in order (as dataset schemas can differ).
if len(dataframes) > 0:
    df = dataframes[record_set_id]
    possible_numeric_fields = ['cr:field/coverage_percentage', 'cr:field/molecular_weight', 'cr:field/peptide_count', 'cr:field/abundance', 'cr:field/normalized_abundance']
    numeric_field = None
    for cand in possible_numeric_fields:
        if cand in df.columns:
            numeric_field = cand
            break
    if numeric_field is None:
        # Fallback to first numeric column if available
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    print(f"Using numeric field: {numeric_field}")

    threshold = 10
    if numeric_field is not None:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field
        possible_group_fields = ['cr:field/description', 'cr:field/peptide_sequence', 'cr:field/accession', 'cr:field/sample']
        group_field = None
        for g in possible_group_fields:
            if g in df.columns:
                group_field = g
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (showing means of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group_field found in the DataFrame.")
    else:
        print("No numeric field detected in this record set. Skipping EDA.")
else:
    print('DataFrame is not available for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field and, if possible, the relationship between two features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If there's a group field, show boxplot
    if group_field is not None:
        plt.figure(figsize=(10,5))
        # Show only top 12 categories for clarity
        if df[group_field].nunique() < 13:
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xticks(rotation=45)
            plt.show()
        else:
            top_cats = df[group_field].value_counts().index[:10]
            sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_cats)])
            plt.title(f'{numeric_field} by top 10 {group_field} categories')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
- **Explored** the FAIR² Croissant dataset metadata and discovered available record sets and fields by their `@id`.
- **Loaded** records into pandas DataFrames for analysis.
- **Performed basic EDA:** filtering, normalization, and grouping by key fields (referenced by `@id`).
- **Visualized** numeric distributions with histograms and boxplots.

> All manipulation and field access was performed using the Croissant schema's `@id` convention for transparency and reproducibility.

### Next Steps
- Engineer biological or domain-specific features for advanced ML or statistical analysis.
- Integrate with protein databases (e.g., UniProt) using accession numbers for enrichment.
- Develop downstream ML pipelines compliant with the Croissant data lineage.

----
_Notebook powered by [`mlcroissant`](https://mlcroissant.readthedocs.io/). For schema and dataset details, see the [FAIR² package landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa)._